### load the pair with proper indexing at line 2

## Load the json files that are created in plot2_overleaf ICSD.ipynb 
### The file names are like : sg_icsd_pair_1-12.json ,...,

In [ ]:
f = glob.glob('sg_icsd_*pair*.json')
with open(f[10], 'r') as file:
    df = json.load(file)

'c:\\Users\\moons\\OneDrive\\Documents\\polymorph_revised\\polymorphism'

In [9]:
import os
import json
import glob
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from pymatgen.core import Structure
from pymatgen.io.cif import CifParser
from pymatgen.io.ase import AseAtomsAdaptor
from ase.io import read
from ase.visualize.plot import plot_atoms

# -----------------------
# Config
# -----------------------
# from step1 you know what pairs you got as most frequent, alternatively, look at the image saved from step before

CIF_FOLDER = r"c:\Users\moons\OneDrive\Documents\polymorph_revised\polymorphism\icsd\CIF"  # cif files are stored here
group1 = '221.0'
group2 = '225.0'
# -----------------------
# Load pair JSON
# -----------------------
f = glob.glob('sg_icsd_*pair*.json')
with open(f[10], 'r') as file:
    df = json.load(file)

# -----------------------
# Match formula pairs
# -----------------------
matched_pairs = []

# Precompute reduced formulas for group2
formula_group2 = {}
for cif in df[group2]:
    path = os.path.join(CIF_FOLDER, cif)
    try:
        struct = Structure.from_file(path)
        formula_group2[path] = struct.composition.reduced_formula
    except:
        continue

# Match with group1 entries
for cif1 in df[group1]:
    path1 = os.path.join(CIF_FOLDER, cif1)
    try:
        struct1 = Structure.from_file(path1)
        formula1 = struct1.composition.reduced_formula
    except:
        continue

    for path2, formula2 in formula_group2.items():
        if formula1 == formula2:
            matched_pairs.append((path1, path2))

# -----------------------
# Helper functions
# -----------------------

def to_subscript(formula):
    sub_map = str.maketrans("0123456789", "₀₁₂₃₄₅₆₇₈₉")
    result = ""
    for c in formula:
        result += c.translate(sub_map) if c.isdigit() else c
    return result

def get_reduced_formula(cif_path):
    try:
        struct = CifParser(cif_path).get_structures()[0]
        return struct.composition.reduced_formula
    except:
        return "Unknown"

def load_structure(filename):
    try:
        return read(filename)
    except:
        try:
            struct = CifParser(filename).get_structures()[0]
            return AseAtomsAdaptor.get_atoms(struct)
        except:
            print("skipped:", filename.split("/")[-1])
            return None

def plot_cif_pair(cif1_path, cif2_path, outname):
    atoms1 = load_structure(cif1_path)
    atoms2 = load_structure(cif2_path)

    if atoms1 is None or atoms2 is None:
        return

    formula1 = to_subscript(get_reduced_formula(cif1_path))
    formula2 = to_subscript(get_reduced_formula(cif2_path))

    os.makedirs("overleaf_icsd_sg_pair", exist_ok=True)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    plot_atoms(atoms1, ax=axes[0], rotation=('20x,10y'), show_unit_cell=2)
    plot_atoms(atoms2, ax=axes[1], rotation=('20x,10y'), show_unit_cell=2)

    mpid1 = os.path.basename(cif1_path).split(".")[0]
    mpid2 = os.path.basename(cif2_path).split(".")[0]

    axes[0].set_title(f"{mpid1} {formula1}", fontsize=23)
    axes[1].set_title(f"{mpid2} {formula2}", fontsize=23)
    # axes[0].tick_params(labelsize=25)
    # axes[1].tick_params(labelsize=25)
    plt.tight_layout()
    plt.savefig(f"overleaf_icsd_sg_pair/{outname}.png", dpi=300)
    plt.close()

# -----------------------
# Generate image outputs
# -----------------------
# Only save specific matched pairs
allowed_names = {'mp-1183151_mp-1095936.png', 'mp-1183165_mp-1093918.png', 'mp-1183187_mp-1096207.png'}
# allowed_names={'pair_6_151385_608800.png','pair_70_53461_53462.png','pair_4_181910_150823.png'}
for cif1, cif2 in matched_pairs:
    name1 = os.path.splitext(os.path.basename(cif1))[0]
    name2 = os.path.splitext(os.path.basename(cif2))[0]
    if name1 == name2:
        continue  # Skip identical names
    outname = f"{name1}_{name2}"
    
    # Check if this pair should be saved
    if f"{outname}.png" not in allowed_names:
        plot_cif_pair(cif1, cif2, outname)

c:\Users\moons\anaconda3\Lib\site-packages\pymatgen\io\cif.py:1312: UserWarning: Cannot determine chemical composition from CIF! could not convert string to float: '2-'
  if struct := self._get_structure(data, primitive, symmetrized, check_occu=check_occu):
c:\Users\moons\anaconda3\Lib\site-packages\pymatgen\io\cif.py:1312: UserWarning: Cannot determine chemical composition from CIF! + + is an invalid formula!
  if struct := self._get_structure(data, primitive, symmetrized, check_occu=check_occu):
c:\Users\moons\anaconda3\Lib\site-packages\pymatgen\io\cif.py:1312: UserWarning: Cannot determine chemical composition from CIF! could not convert string to float: '1-'
  if struct := self._get_structure(data, primitive, symmetrized, check_occu=check_occu):
c:\Users\moons\anaconda3\Lib\site-packages\pymatgen\io\cif.py:1312: UserWarning: Cannot determine chemical composition from CIF! could not convert string to float: '3-'
  if struct := self._get_structure(data, primitive, symmetrized, check

In [11]:
import pandas as pd
import os
os.getcwd()

'c:\\Users\\moons\\OneDrive\\Documents\\polymorph_revised\\polymorphism'

In [12]:
d=pd.read_csv('dataset/icsd_deduped3.csv')
d.shape

(15461, 17)

In [14]:
d['reduced_formula'].value_counts().head(15)

reduced_formula
SiO2               62
CdI2               41
C                  22
CuI                20
BN                 18
ZnS                16
InSb               15
NaH                14
Bi2O3              14
Ba                 13
Ba2YCu3O7          13
Li4CO4             13
O2                 12
Al4.5Si1.5O9.75    12
Rb4CO4             12
Name: count, dtype: int64

In [18]:
d2=pd.read_csv('dataset/icsd_dataset.csv')

d2['reduced_formula'].value_counts().head(15)

reduced_formula
SiO2         558
ZnS          231
CdI2         214
MgAl2O4      190
TiO2         166
Mg(FeO2)2    134
CaCO3        105
AlPO4        100
FeSe          99
BaTiO3        94
ZrO2          92
MgSiO3        92
CuI           91
Al2NiO4       88
Mg2SiO4       83
Name: count, dtype: int64

In [24]:
d3=d2[d2['reduced_formula']=='MgSi2']['filename'].tolist()
d3

[]

In [22]:
CIF_FOLDER = r"c:\Users\moons\OneDrive\Documents\polymorph_revised\polymorphism\icsd\CIF"

for filename in d3:
    path = os.path.join(CIF_FOLDER, filename)
    try:
        struct = Structure.from_file(path)
        print(f"{filename}: {len(struct)} atoms")
    except Exception as e:
        print(f"{filename}: Error loading - {e}")

100081.cif: 48 atoms
100199.cif: 72 atoms
100279.cif: 48 atoms
100341.cif: 9 atoms
100342.cif: 9 atoms
100343.cif: 9 atoms
100344.cif: 9 atoms
100345.cif: 9 atoms
100346.cif: 9 atoms
100749.cif: 48 atoms
100750.cif: 48 atoms
100751.cif: 48 atoms
100752.cif: 48 atoms
100753.cif: 48 atoms
100754.cif: 48 atoms
100755.cif: 48 atoms
10078.cif: 6 atoms
109195.cif: 6 atoms
1109.cif: 144 atoms


c:\Users\moons\anaconda3\Lib\site-packages\pymatgen\core\structure.py:3087: UserWarning: Issues encountered while parsing CIF: 1 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]


1440.cif: 960 atoms
151108.cif: 36 atoms
152295.cif: 54 atoms
152448.cif: 324 atoms
15321.cif: 24 atoms
153257.cif: 96 atoms
153334.cif: 216 atoms
153336.cif: 432 atoms
153470.cif: 72 atoms
153471.cif: 144 atoms
153472.cif: 144 atoms
153886.cif: 12 atoms
154289.cif: 9 atoms
154321.cif: 12 atoms
155241.cif: 9 atoms
155242.cif: 9 atoms
155243.cif: 9 atoms
155244.cif: 12 atoms
155245.cif: 12 atoms
155246.cif: 12 atoms
155247.cif: 9 atoms
155248.cif: 9 atoms
155249.cif: 9 atoms
155250.cif: 9 atoms
155251.cif: 9 atoms
155252.cif: 9 atoms
155684.cif: 72 atoms
153470.cif: 72 atoms
153471.cif: 144 atoms
153472.cif: 144 atoms
153886.cif: 12 atoms
154289.cif: 9 atoms
154321.cif: 12 atoms
155241.cif: 9 atoms
155242.cif: 9 atoms
155243.cif: 9 atoms
155244.cif: 12 atoms
155245.cif: 12 atoms
155246.cif: 12 atoms
155247.cif: 9 atoms
155248.cif: 9 atoms
155249.cif: 9 atoms
155250.cif: 9 atoms
155251.cif: 9 atoms
155252.cif: 9 atoms
155684.cif: 72 atoms
156195.cif: 48 atoms
156196.cif: 9 atoms
156197.c

c:\Users\moons\anaconda3\Lib\site-packages\pymatgen\core\structure.py:3087: UserWarning: Issues encountered while parsing CIF: 4 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]


162629.cif: 48 atoms
162630.cif: 48 atoms
162631.cif: 6 atoms
162632.cif: 6 atoms
162660.cif: 24 atoms
163224.cif: 288 atoms
163225.cif: 288 atoms
16331.cif: 9 atoms
16332.cif: 9 atoms
16333.cif: 9 atoms
16334.cif: 9 atoms
16335.cif: 9 atoms
16336.cif: 9 atoms
164159.cif: 192 atoms
164160.cif: 168 atoms
164161.cif: 168 atoms
164162.cif: 168 atoms
165687.cif: 252 atoms
165878.cif: 168 atoms
166601.cif: 9 atoms
166602.cif: 9 atoms
166603.cif: 9 atoms
167786.cif: 108 atoms
168350.cif: 9 atoms
168351.cif: 9 atoms
168352.cif: 9 atoms
168353.cif: 9 atoms
168354.cif: 9 atoms
164160.cif: 168 atoms
164161.cif: 168 atoms
164162.cif: 168 atoms
165687.cif: 252 atoms
165878.cif: 168 atoms
166601.cif: 9 atoms
166602.cif: 9 atoms
166603.cif: 9 atoms
167786.cif: 108 atoms
168350.cif: 9 atoms
168351.cif: 9 atoms
168352.cif: 9 atoms
168353.cif: 9 atoms
168354.cif: 9 atoms


c:\Users\moons\anaconda3\Lib\site-packages\pymatgen\core\structure.py:3087: UserWarning: Issues encountered while parsing CIF: 10 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]


168355.cif: 9 atoms
170475.cif: 144 atoms
170476.cif: 96 atoms
170477.cif: 288 atoms
170478.cif: 576 atoms
170479.cif: 288 atoms
170480.cif: 96 atoms
170481.cif: 36 atoms
170482.cif: 144 atoms
170483.cif: 144 atoms
170484.cif: 288 atoms
170485.cif: 288 atoms
170486.cif: 108 atoms
170479.cif: 288 atoms
170480.cif: 96 atoms
170481.cif: 36 atoms
170482.cif: 144 atoms
170483.cif: 144 atoms
170484.cif: 288 atoms
170485.cif: 288 atoms
170486.cif: 108 atoms
170487.cif: 144 atoms
170488.cif: 144 atoms
170489.cif: 36 atoms
170490.cif: 72 atoms
170492.cif: 108 atoms
170493.cif: 72 atoms
170494.cif: 72 atoms
170495.cif: 36 atoms
170496.cif: 18 atoms
170497.cif: 48 atoms
170498.cif: 96 atoms
170499.cif: 96 atoms
170500.cif: 24 atoms
170501.cif: 96 atoms
170487.cif: 144 atoms
170488.cif: 144 atoms
170489.cif: 36 atoms
170490.cif: 72 atoms
170492.cif: 108 atoms
170493.cif: 72 atoms
170494.cif: 72 atoms
170495.cif: 36 atoms
170496.cif: 18 atoms
170497.cif: 48 atoms
170498.cif: 96 atoms
170499.cif: 96

c:\Users\moons\anaconda3\Lib\site-packages\pymatgen\core\structure.py:3087: UserWarning: Issues encountered while parsing CIF: 3 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
c:\Users\moons\anaconda3\Lib\site-packages\pymatgen\core\structure.py:3087: UserWarning: Issues encountered while parsing CIF: 2 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]


170527.cif: 48 atoms
170528.cif: 96 atoms
170529.cif: 48 atoms
170530.cif: 48 atoms
170531.cif: 48 atoms
170532.cif: 24 atoms
170533.cif: 24 atoms
170534.cif: 48 atoms
170535.cif: 24 atoms
170536.cif: 24 atoms
170537.cif: 48 atoms
170538.cif: 24 atoms
170539.cif: 54 atoms
170540.cif: 54 atoms
170541.cif: 48 atoms
170543.cif: 72 atoms
170544.cif: 54 atoms
170545.cif: 48 atoms
170546.cif: 24 atoms
170547.cif: 48 atoms
170548.cif: 12 atoms
170549.cif: 24 atoms
170550.cif: 24 atoms
170551.cif: 24 atoms
170552.cif: 18 atoms
170554.cif: 36 atoms
170559.cif: 72 atoms
170561.cif: 72 atoms
170997.cif: 360 atoms
171573.cif: 9 atoms
171733.cif: 48 atoms
170544.cif: 54 atoms
170545.cif: 48 atoms
170546.cif: 24 atoms
170547.cif: 48 atoms
170548.cif: 12 atoms
170549.cif: 24 atoms
170550.cif: 24 atoms
170551.cif: 24 atoms
170552.cif: 18 atoms
170554.cif: 36 atoms
170559.cif: 72 atoms
170561.cif: 72 atoms
170997.cif: 360 atoms
171573.cif: 9 atoms
171733.cif: 48 atoms
171734.cif: 72 atoms
171735.cif: 9